ADD EXCEPTIONS AND WHILE LOOP IN ALL PROCESSES THAT INVOLVES USER INPUT
    
data import - ✅
check if the uploaded dataset is a xlxs or csv file - ✅
give a summary of the dataset - ✅
remove any outliers ( winzorisation ) - ✅ 
checking for duplicates and removing - ✅ 
input target seperation - ✅ 
check if the target column is Numeric, binary or categorical
if regression:
    data type =  int/ float
    unique values >20
    unique ratio >0.5
    user confirmation
if classification:
    data type = object, Int
    unique values < 20
    unique ratio < 0.5
    if object : 
        encode them into integer and create a dictinary to store matching values
    user confirmation
if its object and has 20 + unique values - say the target column is invalid

ask for the ML Type ( regression or classification )
if regression:
    convertion of categorical data into numeric
    use feature importance to remove cols if theres too much
    split into train and test data 
    checking of missing data and filling it
    use all the regressors provided 
    find the best model
    provide the the model through joblib
    ask input if theres a need to predict
    thanks

if classifier:
    check for class imbalance 
    convertion of categorical data into numeric
    use feature importance to remove cols if theres too much
    checking of missing data and filling it
    standardisation ( standard scaler)
    split into train and test data ( stratified k fold ) 
    if class imbalance:
        use Smote on training data alone to stabilize
    use all the classifiers provided 
    find the best model
    provide the the model and scaler through joblib
    ask input if theres a need to predict
    thanks

filepath = "D:\COLLEGE\heart.csv"

In [17]:
#IMPORTING THE NECESSARY LIBRARIES..!
import pandas as pd
import numpy as np

In [2]:
# DATA IMPORT MODULE..!

def data_ingestion(): 
    data = None
    while data is None:
        try:
            file_path = input ("Give the path to the dataset that you want to be processed (enclose it within quotes):").strip(" \" ")
            
            if file_path.endswith(".csv") is True:
                data = pd.read_csv(file_path)
            elif file_path.endswith(".xlsx") is True:
                data = pd.read_excel(file_path)
            else:
                print("The file needs to be in .csv or .xlsx format")
                continue
                
        except FileNotFoundError:
            print("❌ File not found. Please check the path.")
        except PermissionError:
            print("❌ Permission denied while accessing the file.")
        except AttributeError:
            print("Kindly Enter the correct file path again..!")
        else:
            print(f"\n\nThe Dataset has been successfully uploaded, here's a short preview..!\n\n\n{data.head()}")
    return data
    
data = data_ingestion()

Give the path to the dataset that you want to be processed (enclose it within quotes): "D:\COLLEGE\heart.csv"




The Dataset has been successfully uploaded, here's a short preview..!


   Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  MaxHR  \
0   40   M           ATA        140          289          0     Normal    172   
1   49   F           NAP        160          180          0     Normal    156   
2   37   M           ATA        130          283          0         ST     98   
3   48   F           ASY        138          214          0     Normal    108   
4   54   M           NAP        150          195          0     Normal    122   

  ExerciseAngina  Oldpeak ST_Slope  HeartDisease  
0              N      0.0       Up             0  
1              N      1.0     Flat             1  
2              N      0.0       Up             0  
3              Y      1.5     Flat             1  
4              N      0.0       Up             0  


In [4]:
# Gives a description about data..!
def summary_of_data(data):
    columns = data.columns.to_list()
    shape = data.shape
    missing_values = data.isnull().sum()
    duplicates = data.duplicated().sum()

    print(f"The dataset has {shape[0]} rows and {shape[1]} columns.")
    print(f"\nThe column names are : {columns}")
    print(f"\nThe missing values in each attributes are shown below...!\n\n{missing_values}")
    print(F"\n The dataset has {duplicates} duplicate values.")
    print(F" Other things you may want a look into : \n\n{data.describe()}")
summary_of_data(data)

The dataset has 918 rows and 12 columns.

The column names are : ['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS', 'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope', 'HeartDisease']

The missing values in each attributes are shown below...!

Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64

 The dataset has 0 duplicate values.
 Other things you may want a look into : 

              Age   RestingBP  Cholesterol   FastingBS       MaxHR  \
count  918.000000  918.000000   918.000000  918.000000  918.000000   
mean    53.510893  132.396514   198.799564    0.233115  136.809368   
std      9.432617   18.514154   109.384145    0.423046   25.460334   
min     28.000000    0.000000     0.000000    0.000000   60.000000   
25%     47.000000  120.000000   173.25

In [8]:
#OUTLIER HANDLING..!

def handling_outliers(df):
    numerical_columns = df.select_dtypes(include=['number']).columns
    for col in numerical_columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
    print(" The Outliers in the dataset have been successfully dealt with by DataForge( if there was any..! )")
        
    return df
data_without_outliers = handling_outliers(data)


 The Outliers in the dataset have been successfully dealt with ( if there was any..! )


In [12]:
#DUPLICATE REMOVAL..!
def remove_duplicates(df):
    initial_count = len(df)
    df = df.drop_duplicates(keep = "first").reset_index(drop=True)
    final_count = len(df)
    print(f"DataForge Removed {initial_count - final_count} duplicate rows.")
    
    return df

cleaned_data = remove_duplicates(data_without_outliers)

DataForge Removed 0 duplicate rows.


In [13]:
#INPUT TARGET SEPERATION...!
def input_output_separator(df):
    target = None
    while target not in df.columns.to_list():
        target = input ("\nNow for the next part, Correctly type the output column's name :").strip("\"")
        if target not in df.columns.to_list(): 
            print("\nThe input you have given doesnt seem to match any attributes in the dataset, kindly enter it again..!")
        
    input_data = df.columns.to_list()
    input_data.remove(target)
    input_data = df[input_data]
    target = df[[target]]
    print("DataForge has successfully seperated the dataset into input and target data..!")
    return input_data, target
    
input_data, target = input_output_separator(cleaned_data)


Now for the next part, Correctly type the output column's name : HeartDiseaseee



The input you have given doesnt seem to match any attributes in the dataset, kindly enter it again..!



Now for the next part, Correctly type the output column's name : HeartDisease


DataForge has successfully seperated the dataset into input and target data..!


In [15]:
# TARGET TESTING AND MODEL SUITABILITY MODULE..!
def target_tester(target):
    model = ""
    while model not in ["r", "c"]:
        model = input(
            "Enter the type of ML algorithm you want to perform "
            "(Regression / Classification) : (R / C): ").lower()
        if model not in ["r", "c"]:
            print("Please enter a valid input (R or C):")
            
    target_dtype = target.dtype
    unique_num = target.nunique()
    unique_ratio = unique_num / len(target)

    is_numeric = np.issubdtype(target_dtype, np.number)
    is_object = target_dtype == "object"

    if model == "r":
        if is_numeric and unique_num > 20 and unique_ratio > 0.1:
            print("DataForge will now proceed to process the data for ML processes")
        else:
            confirmation = ""
            while confirmation not in ["y", "n"]:
                confirmation = input(
                    "⚠️ Target may be better suited for Classification. "
                    "Do you want to switch? (Y/N): "
                ).lower()
            if confirmation == "y":
                model = "c"

    if model == "c":
        if (is_object) or (is_numeric and unique_num <= 20 and unique_ratio < 0.05):
            print("DataForge will now proceed to process the data for ML processes")
        else:
            confirmation = ""
            while confirmation not in ["y", "n"]:
                confirmation = input(
                    "⚠️ Target may be better suited for Regression. "
                    "Do you want to switch? (Y/N): "
                ).lower()
            if confirmation == "y":
                model = "r"

    return model


IndentationError: expected an indented block (1266369956.py, line 19)

In [ ]:
check if the target column is Numeric, binary or categorical
if regression:
    data type =  int/ float
    unique values >20
    unique ratio >0.5
    user confirmation
if classification:
    data type = object, Int
    unique values < 20
    unique ratio < 0.5
    if object : 
        encode them into integer and create a dictinary to store matching values
    user confirmation
if its object and has 20 + unique values - say the target column is invalid